# 📊 VQA-E: Evaluation & Analysis Notebook

## Đánh giá chi tiết 4 VQA Models — Visual Question Answering with Explanation

**Notebook này chuyên để đánh giá, phân tích kết quả** sau khi đã train xong cả 4 models.  
Chạy trên **Google Colab** — data & checkpoints đã upload sẵn trên Drive/Colab.

### Nội dung notebook:

| # | Section | Mô tả |
|---|---------|--------|
| 0 | **Setup** | Mount Drive, install deps, load code |
| 1 | **Load Models** | Load 4 models từ checkpoints |
| 2 | **Quantitative Eval** | BLEU-1/2/3/4, METEOR, BERTScore, Exact Match |
| 3 | **Comparison Table** | So sánh side-by-side 4 models |
| 4 | **Greedy vs Beam Search** | So sánh decode strategies |
| 5 | **Training Curves** | Loss curves, convergence analysis |
| 6 | **Qualitative Analysis** | Sample predictions, image + answers |
| 7 | **Attention Visualization** | Heatmaps cho Model C/D |
| 8 | **Error Analysis** | Phân tích lỗi, answer length, question type |
| 9 | **Statistical Tests** | Paired t-test, confidence intervals |
| 10 | **Summary & Conclusions** | Tổng hợp kết quả cuối cùng |

### 4 Model variants:
| Model | CNN | Attention | Pretrained |
|-------|-----|-----------|------------|
| **A** | SimpleCNN (scratch) | ❌ | ❌ |
| **B** | ResNet101 (frozen→unfreeze) | ❌ | ✅ ImageNet |
| **C** | SimpleCNN Spatial (49 regions) | ✅ Dual Attn + Coverage | ❌ |
| **D** | ResNet101 Spatial (49 regions) | ✅ Dual Attn + Coverage | ✅ ImageNet |

---
## 0. Environment Setup (Colab)

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ══════════════════════════════════════════════════════════════════════════════
# 📁 CẤU HÌNH ĐƯỜNG DẪN — CHỈNH SỬA Ở ĐÂY
# ══════════════════════════════════════════════════════════════════════════════
# Đường dẫn tới thư mục project trên Google Drive
PROJECT_ROOT = "/content/drive/MyDrive/vqa_new"  # ← Chỉnh sửa nếu khác
# ══════════════════════════════════════════════════════════════════════════════

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q nltk bert-score matplotlib seaborn pandas tqdm pillow scipy

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

In [ ]:
# ── Setup paths & imports ─────────────────────────────────────────────────────
import os, sys, json, torch, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from PIL import Image
from torchvision import transforms
from collections import Counter, defaultdict
from tqdm import tqdm
from IPython.display import display, HTML

# Set project root
os.chdir(PROJECT_ROOT)
sys.path.insert(0, 'src')
print(f"Working directory: {os.getcwd()}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Plot style
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
sns.set_style('whitegrid')

print("\n✅ Setup complete!")

In [ ]:
# ── Paths configuration ───────────────────────────────────────────────────────
VAL_IMAGE_DIR  = "data/raw/val2014"
VAL_VQA_E_JSON = "data/vqa_e/VQA-E_val_set.json"
VOCAB_Q_PATH   = "data/processed/vocab_questions.json"
VOCAB_A_PATH   = "data/processed/vocab_answers.json"
CHECKPOINTS_DIR = "checkpoints"

# Verify paths exist
for p, desc in [(VAL_IMAGE_DIR, 'Val images'), (VAL_VQA_E_JSON, 'VQA-E val JSON'),
                (VOCAB_Q_PATH, 'Question vocab'), (VOCAB_A_PATH, 'Answer vocab'),
                (CHECKPOINTS_DIR, 'Checkpoints')]:
    exists = os.path.exists(p)
    icon = '✅' if exists else '❌'
    print(f"  {icon} {desc}: {p}")
    if not exists:
        print(f"     ⚠ WARNING: Path not found! Check PROJECT_ROOT setting.")

---
## 1. Load Vocab & Models

In [ ]:
# ── Load vocabularies ─────────────────────────────────────────────────────────
from vocab import Vocabulary
from inference import load_model_from_checkpoint
from inference import (
    greedy_decode, greedy_decode_with_attention,
    batch_greedy_decode, batch_greedy_decode_with_attention,
    beam_search_decode, beam_search_decode_with_attention,
    batch_beam_search_decode, batch_beam_search_decode_with_attention,
)

vocab_q = Vocabulary(); vocab_q.load(VOCAB_Q_PATH)
vocab_a = Vocabulary(); vocab_a.load(VOCAB_A_PATH)

print(f"Question vocab size: {len(vocab_q)}")
print(f"Answer vocab size  : {len(vocab_a)}")

In [ ]:
# ── Load all trained models ───────────────────────────────────────────────────
loaded_models = {}
model_info = {}

for m in 'ABCD':
    # Try best checkpoint first, then epoch-specific
    ckpt_candidates = [
        f'{CHECKPOINTS_DIR}/model_{m.lower()}_best.pth',
        f'{CHECKPOINTS_DIR}/model_{m.lower()}_epoch30.pth',
        f'{CHECKPOINTS_DIR}/model_{m.lower()}_epoch10.pth',
    ]
    ckpt = None
    for c in ckpt_candidates:
        if os.path.exists(c):
            ckpt = c
            break

    if ckpt:
        # Auto-detects coverage from state_dict keys
        model = load_model_from_checkpoint(
            m, ckpt, len(vocab_q), len(vocab_a), device=DEVICE
        )
        loaded_models[m] = model

        # Count parameters
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        ckpt_size = os.path.getsize(ckpt) / 1e6
        model_info[m] = {
            'checkpoint': ckpt,
            'total_params': total_params,
            'trainable_params': trainable_params,
            'ckpt_size_mb': ckpt_size,
        }
        print(f"  ✅ Model {m}: {ckpt} ({total_params/1e6:.1f}M params, {ckpt_size:.1f} MB)")
    else:
        print(f"  ❌ Model {m}: No checkpoint found")

print(f"\n  {len(loaded_models)} models loaded: {list(loaded_models.keys())}")

In [ ]:
# ── Model Architecture Summary ───────────────────────────────────────────────
MODEL_DESCRIPTIONS = {
    'A': 'SimpleCNN (scratch), No Attention',
    'B': 'ResNet101 (pretrained), No Attention',
    'C': 'SimpleCNN Spatial, Dual Attention + Coverage',
    'D': 'ResNet101 Spatial, Dual Attention + Coverage',
}

rows = []
for m in 'ABCD':
    if m in model_info:
        info = model_info[m]
        rows.append({
            'Model': f'Model {m}',
            'Architecture': MODEL_DESCRIPTIONS[m],
            'Total Params (M)': f"{info['total_params']/1e6:.1f}",
            'Checkpoint Size (MB)': f"{info['ckpt_size_mb']:.1f}",
            'Checkpoint': os.path.basename(info['checkpoint']),
        })

df_arch = pd.DataFrame(rows)
display(df_arch.style.set_caption('Model Architecture Summary').hide(axis='index'))

---
## 2. Quantitative Evaluation — Full Metrics

Đánh giá từng model trên **validation set** với các metrics:

| Metric | Loại | Ý nghĩa |
|--------|------|----------|
| **BLEU-4** ★ | N-gram overlap | Precision-based, penalizes short output |
| **METEOR** ★ | Alignment-based | Handles synonyms, stemming |
| **BERTScore** ★ | Semantic | Cosine similarity of BERT embeddings |
| BLEU-1/2/3 | N-gram overlap | Unigram → trigram precision |
| Exact Match | String compare | Chính xác 100% |

★ = Primary metrics

In [ ]:
# ── Evaluation Helper Functions ───────────────────────────────────────────────
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as nltk_meteor
from torch.utils.data import DataLoader
from dataset import VQAEDataset, vqa_collate_fn

try:
    from bert_score import score as bert_score_fn
    HAS_BERTSCORE = True
    print("✅ BERTScore available")
except ImportError:
    HAS_BERTSCORE = False
    print("⚠ BERTScore not available (pip install bert-score)")


def decode_tensor(a_tensor, vocab_a):
    """Convert answer tensor to string, stripping special tokens."""
    special = {
        vocab_a.word2idx['<pad>'],
        vocab_a.word2idx['<start>'],
        vocab_a.word2idx['<end>']
    }
    words = [vocab_a.idx2word[int(i)] for i in a_tensor if int(i) not in special]
    return ' '.join(words)


def evaluate_model(model, model_type, val_dataset, vocab_a,
                   beam_width=1, no_repeat_ngram_size=3, batch_size=64):
    """
    Evaluate a single model, return dict of metrics + per-sample predictions.
    """
    use_attention = model_type in ('C', 'D')

    if beam_width > 1:
        decode_fn = (
            batch_beam_search_decode_with_attention if use_attention
            else batch_beam_search_decode
        )
        decode_kwargs = dict(beam_width=beam_width,
                             no_repeat_ngram_size=no_repeat_ngram_size)
    else:
        decode_fn = (
            batch_greedy_decode_with_attention if use_attention
            else batch_greedy_decode
        )
        decode_kwargs = {}

    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=vqa_collate_fn, num_workers=2
    )

    smoothie = SmoothingFunction().method1
    all_predictions = []
    all_gt_strings  = []

    with torch.no_grad():
        for imgs, questions, answers in tqdm(val_loader, desc=f"Model {model_type}", leave=False):
            preds = decode_fn(model, imgs, questions, vocab_a, device=DEVICE, **decode_kwargs)
            all_predictions.extend(preds)
            for a_tensor in answers:
                all_gt_strings.append(decode_tensor(a_tensor, vocab_a))

    n = len(all_predictions)
    exact_match = 0
    bleu1_total, bleu2_total, bleu3_total, bleu4_total = 0.0, 0.0, 0.0, 0.0
    meteor_total = 0.0
    per_sample = []  # for detailed analysis later

    for pred_str, gt_str in zip(all_predictions, all_gt_strings):
        pred_clean = pred_str.strip().lower()
        gt_clean   = gt_str.strip().lower()

        em = 1 if pred_clean == gt_clean else 0
        exact_match += em

        gt_words   = gt_str.split() or ['<unk>']
        pred_words = pred_str.split() or ['<unk>']

        b1 = sentence_bleu([gt_words], pred_words, weights=(1, 0, 0, 0), smoothing_function=smoothie)
        b2 = sentence_bleu([gt_words], pred_words, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie)
        b3 = sentence_bleu([gt_words], pred_words, weights=(1/3, 1/3, 1/3, 0), smoothing_function=smoothie)
        b4 = sentence_bleu([gt_words], pred_words, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)
        met = nltk_meteor([gt_words], pred_words)

        bleu1_total += b1
        bleu2_total += b2
        bleu3_total += b3
        bleu4_total += b4
        meteor_total += met

        per_sample.append({
            'prediction': pred_str,
            'ground_truth': gt_str,
            'exact_match': em,
            'bleu1': b1, 'bleu2': b2, 'bleu3': b3, 'bleu4': b4,
            'meteor': met,
            'pred_len': len(pred_words),
            'gt_len': len(gt_words),
        })

    # BERTScore
    bertscore_f1 = 0.0
    if HAS_BERTSCORE:
        print(f"  Computing BERTScore for Model {model_type}...")
        _, _, F1 = bert_score_fn(all_predictions, all_gt_strings, lang='en', verbose=False)
        bertscore_f1 = F1.mean().item()
        # Add per-sample BERTScore
        for i, f1 in enumerate(F1.tolist()):
            per_sample[i]['bertscore'] = f1

    metrics = {
        'model_type':  model_type,
        'bleu1':       bleu1_total / n,
        'bleu2':       bleu2_total / n,
        'bleu3':       bleu3_total / n,
        'bleu4':       bleu4_total / n,
        'meteor':      meteor_total / n,
        'bertscore':   bertscore_f1,
        'exact_match': exact_match / n * 100,
        'n_samples':   n,
    }

    return metrics, per_sample

In [ ]:
# ── Load validation dataset ───────────────────────────────────────────────────
# Dùng NUM_EVAL_SAMPLES = None cho full set, hoặc số nhỏ hơn để test nhanh
NUM_EVAL_SAMPLES = 2000  # ← Chỉnh None cho full set (~30k samples)

val_dataset = VQAEDataset(
    image_dir=VAL_IMAGE_DIR,
    vqa_e_json_path=VAL_VQA_E_JSON,
    vocab_q=vocab_q,
    vocab_a=vocab_a,
    split='val2014',
    max_samples=NUM_EVAL_SAMPLES
)
print(f"Validation samples: {len(val_dataset)}")

In [ ]:
# ── Evaluate all models (Greedy Decode) ───────────────────────────────────────
all_metrics_greedy = {}
all_per_sample_greedy = {}

for m in sorted(loaded_models.keys()):
    print(f"\n{'='*50}")
    print(f"  Evaluating Model {m} (Greedy)")
    print(f"{'='*50}")
    metrics, per_sample = evaluate_model(
        loaded_models[m], m, val_dataset, vocab_a, beam_width=1
    )
    all_metrics_greedy[m] = metrics
    all_per_sample_greedy[m] = per_sample

    # Print results
    print(f"  BLEU-4  ★ : {metrics['bleu4']:.4f}")
    print(f"  METEOR  ★ : {metrics['meteor']:.4f}")
    if HAS_BERTSCORE:
        print(f"  BERTScore : {metrics['bertscore']:.4f}")
    print(f"  BLEU-1    : {metrics['bleu1']:.4f}")
    print(f"  Exact Match: {metrics['exact_match']:.2f}%")

print("\n✅ Greedy evaluation complete!")

---
## 3. Comparison Table — All Models

In [ ]:
# ── Comparison DataFrame (Greedy) ─────────────────────────────────────────────
rows = []
for m in 'ABCD':
    if m in all_metrics_greedy:
        r = all_metrics_greedy[m]
        rows.append({
            'Model': f'Model {m}',
            'Architecture': MODEL_DESCRIPTIONS[m],
            'BLEU-4 ★': f"{r['bleu4']:.4f}",
            'METEOR ★': f"{r['meteor']:.4f}",
            'BERTScore': f"{r['bertscore']:.4f}" if HAS_BERTSCORE else 'N/A',
            'BLEU-1': f"{r['bleu1']:.4f}",
            'BLEU-2': f"{r['bleu2']:.4f}",
            'BLEU-3': f"{r['bleu3']:.4f}",
            'Exact Match': f"{r['exact_match']:.2f}%",
        })

df_greedy = pd.DataFrame(rows)
print("╔══════════════════════════════════════════════════════════════╗")
print("║          GREEDY DECODE — All Models Comparison             ║")
print("╚══════════════════════════════════════════════════════════════╝")
display(df_greedy.style.set_caption(f'Greedy Decode Results ({NUM_EVAL_SAMPLES or "full"} samples)').hide(axis='index'))

In [ ]:
# ── Bar Chart — Primary Metrics ───────────────────────────────────────────────
model_names = [f'Model {m}' for m in sorted(all_metrics_greedy.keys())]
colors = {'A': '#1f77b4', 'B': '#ff7f0e', 'C': '#2ca02c', 'D': '#d62728'}
bar_colors = [colors[m] for m in sorted(all_metrics_greedy.keys())]

metrics_to_plot = ['bleu4', 'meteor']
if HAS_BERTSCORE:
    metrics_to_plot.append('bertscore')
metrics_to_plot.extend(['bleu1', 'bleu2', 'bleu3'])

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(4 * len(metrics_to_plot), 5))
if len(metrics_to_plot) == 1:
    axes = [axes]

metric_labels = {
    'bleu4': 'BLEU-4 ★', 'meteor': 'METEOR ★', 'bertscore': 'BERTScore ★',
    'bleu1': 'BLEU-1', 'bleu2': 'BLEU-2', 'bleu3': 'BLEU-3',
}

for ax, metric in zip(axes, metrics_to_plot):
    values = [all_metrics_greedy[m][metric] for m in sorted(all_metrics_greedy.keys())]
    bars = ax.bar(model_names, values, color=bar_colors, alpha=0.85, edgecolor='white', linewidth=1.5)
    ax.set_title(metric_labels.get(metric, metric), fontsize=12, fontweight='bold')
    ax.set_ylabel('Score')

    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

fig.suptitle('Model Comparison — Greedy Decode', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('checkpoints/eval_comparison_greedy.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Greedy vs Beam Search Comparison

In [ ]:
# ── Evaluate all models with Beam Search ──────────────────────────────────────
BEAM_WIDTH = 5
NO_REPEAT_NGRAM = 3

all_metrics_beam = {}
all_per_sample_beam = {}

for m in sorted(loaded_models.keys()):
    print(f"\n{'='*50}")
    print(f"  Evaluating Model {m} (Beam Search, width={BEAM_WIDTH})")
    print(f"{'='*50}")
    metrics, per_sample = evaluate_model(
        loaded_models[m], m, val_dataset, vocab_a,
        beam_width=BEAM_WIDTH, no_repeat_ngram_size=NO_REPEAT_NGRAM
    )
    all_metrics_beam[m] = metrics
    all_per_sample_beam[m] = per_sample

    print(f"  BLEU-4  ★ : {metrics['bleu4']:.4f}")
    print(f"  METEOR  ★ : {metrics['meteor']:.4f}")

print("\n✅ Beam search evaluation complete!")

In [ ]:
# ── Greedy vs Beam Search Comparison Table ────────────────────────────────────
rows = []
for m in 'ABCD':
    if m in all_metrics_greedy and m in all_metrics_beam:
        g = all_metrics_greedy[m]
        b = all_metrics_beam[m]
        rows.append({
            'Model': f'Model {m}',
            'BLEU-4 (Greedy)': f"{g['bleu4']:.4f}",
            'BLEU-4 (Beam)': f"{b['bleu4']:.4f}",
            'Δ BLEU-4': f"{b['bleu4'] - g['bleu4']:+.4f}",
            'METEOR (Greedy)': f"{g['meteor']:.4f}",
            'METEOR (Beam)': f"{b['meteor']:.4f}",
            'Δ METEOR': f"{b['meteor'] - g['meteor']:+.4f}",
        })

df_compare = pd.DataFrame(rows)
print("╔══════════════════════════════════════════════════════════════╗")
print(f"║      GREEDY vs BEAM SEARCH (width={BEAM_WIDTH}) Comparison          ║")
print("╚══════════════════════════════════════════════════════════════╝")
display(df_compare.style.set_caption(f'Greedy vs Beam Search (width={BEAM_WIDTH})').hide(axis='index'))

In [ ]:
# ── Grouped Bar Chart: Greedy vs Beam ─────────────────────────────────────────
models_avail = sorted(set(all_metrics_greedy.keys()) & set(all_metrics_beam.keys()))
x = np.arange(len(models_avail))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# BLEU-4
greedy_b4 = [all_metrics_greedy[m]['bleu4'] for m in models_avail]
beam_b4   = [all_metrics_beam[m]['bleu4'] for m in models_avail]
ax1.bar(x - width/2, greedy_b4, width, label='Greedy', color='#4c72b0', alpha=0.85)
ax1.bar(x + width/2, beam_b4, width, label=f'Beam (w={BEAM_WIDTH})', color='#dd8452', alpha=0.85)
ax1.set_title('BLEU-4', fontsize=13, fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels([f'Model {m}' for m in models_avail])
ax1.legend(); ax1.set_ylabel('Score')
for i, (g, b) in enumerate(zip(greedy_b4, beam_b4)):
    ax1.text(i - width/2, g + 0.002, f'{g:.4f}', ha='center', fontsize=8)
    ax1.text(i + width/2, b + 0.002, f'{b:.4f}', ha='center', fontsize=8)

# METEOR
greedy_met = [all_metrics_greedy[m]['meteor'] for m in models_avail]
beam_met   = [all_metrics_beam[m]['meteor'] for m in models_avail]
ax2.bar(x - width/2, greedy_met, width, label='Greedy', color='#4c72b0', alpha=0.85)
ax2.bar(x + width/2, beam_met, width, label=f'Beam (w={BEAM_WIDTH})', color='#dd8452', alpha=0.85)
ax2.set_title('METEOR', fontsize=13, fontweight='bold')
ax2.set_xticks(x); ax2.set_xticklabels([f'Model {m}' for m in models_avail])
ax2.legend(); ax2.set_ylabel('Score')
for i, (g, b) in enumerate(zip(greedy_met, beam_met)):
    ax2.text(i - width/2, g + 0.002, f'{g:.4f}', ha='center', fontsize=8)
    ax2.text(i + width/2, b + 0.002, f'{b:.4f}', ha='center', fontsize=8)

fig.suptitle('Greedy vs Beam Search Decode', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('checkpoints/eval_greedy_vs_beam.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Training Curves Analysis

In [ ]:
# ── Load training history ─────────────────────────────────────────────────────
MODEL_COLORS = {'A': '#1f77b4', 'B': '#ff7f0e', 'C': '#2ca02c', 'D': '#d62728'}
MODEL_LABELS = {
    'A': 'A: SimpleCNN, No Attn',
    'B': 'B: ResNet101, No Attn',
    'C': 'C: SimpleCNN, Dual Attn',
    'D': 'D: ResNet101, Dual Attn',
}
PHASE_BOUNDARIES = [15, 25]  # Phase 1→2, Phase 2→3

histories = {}
for m in 'ABCD':
    hp = f"checkpoints/history_model_{m.lower()}.json"
    if os.path.exists(hp):
        with open(hp) as f:
            histories[m] = json.load(f)
        print(f"  ✅ Model {m}: {len(histories[m]['train_loss'])} epochs")
    else:
        print(f"  ❌ Model {m}: No history file")

In [ ]:
# ── Training & Validation Loss Curves ─────────────────────────────────────────
if histories:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
    ax_train, ax_val, ax_gap = axes

    for m, h in histories.items():
        epochs = list(range(1, len(h['train_loss']) + 1))
        color = MODEL_COLORS[m]
        label = MODEL_LABELS[m]

        # Train loss
        ax_train.plot(epochs, h['train_loss'], color=color, label=label,
                      marker='o', ms=3, lw=1.5)
        # Val loss
        ax_val.plot(epochs, h['val_loss'], color=color, label=label,
                    marker='s', ms=3, ls='--', lw=1.5)
        # Generalization gap (val - train)
        min_len = min(len(h['train_loss']), len(h['val_loss']))
        gap = [h['val_loss'][i] - h['train_loss'][i] for i in range(min_len)]
        ax_gap.plot(epochs[:min_len], gap, color=color, label=label,
                    marker='D', ms=3, lw=1.5)

    for ax, title, ylabel in [
        (ax_train, 'Training Loss', 'Loss'),
        (ax_val, 'Validation Loss', 'Loss'),
        (ax_gap, 'Generalization Gap (Val - Train)', 'Gap')
    ]:
        # Phase boundary lines
        for pe in PHASE_BOUNDARIES:
            ax.axvline(pe, color='gray', ls=':', alpha=0.6, lw=1)
        ax.set_xlabel('Epoch', fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.legend(fontsize=8, loc='best')

        # Phase annotations
        ymax = ax.get_ylim()[1]
        ax.text(7.5,  ymax*0.97, 'Phase 1\n(Base)', ha='center', fontsize=7, color='gray', va='top')
        ax.text(20,   ymax*0.97, 'Phase 2\n(Finetune)', ha='center', fontsize=7, color='gray', va='top')
        ax.text(27.5, ymax*0.97, 'Phase 3\n(SS)', ha='center', fontsize=7, color='gray', va='top')

    ax_gap.axhline(0, color='black', ls='-', lw=0.8, alpha=0.5)

    fig.suptitle('Training Progress — 4 Models × 3 Phases', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('checkpoints/eval_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⚠ No history files found")

In [ ]:
# ── Training Summary Table ────────────────────────────────────────────────────
if histories:
    rows = []
    for m, h in sorted(histories.items()):
        n_epochs = len(h['train_loss'])
        best_val = min(h['val_loss'])
        best_epoch = h['val_loss'].index(best_val) + 1
        final_train = h['train_loss'][-1]
        final_val = h['val_loss'][-1]
        gap = final_val - final_train
        rows.append({
            'Model': f'Model {m}',
            'Epochs': n_epochs,
            'Best Val Loss': f'{best_val:.4f}',
            'Best Epoch': best_epoch,
            'Final Train Loss': f'{final_train:.4f}',
            'Final Val Loss': f'{final_val:.4f}',
            'Gap (Val-Train)': f'{gap:.4f}',
            'Overfitting?': '⚠ Yes' if gap > 0.3 else '✅ OK',
        })
    df_train = pd.DataFrame(rows)
    display(df_train.style.set_caption('Training Summary').hide(axis='index'))

---
## 6. Qualitative Analysis — Sample Predictions

Xem predictions của từng model trên các samples cụ thể, kèm hình ảnh.

In [ ]:
# ── Load validation annotations & prepare samples ─────────────────────────────
val_anns = json.load(open(VAL_VQA_E_JSON))

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Select diverse samples (random seed for reproducibility)
random.seed(42)
samples = random.sample(val_anns, min(20, len(val_anns)))
print(f"Selected {len(samples)} samples for qualitative analysis")

In [ ]:
# ── Visual Predictions: Image + Question + All Models' Answers ─────────────
N_SHOW = 8  # Number of samples to visualize

if not loaded_models:
    print("⚠ No models loaded")
else:
    for idx in range(min(N_SHOW, len(samples))):
        ann    = samples[idx]
        img_id = ann['img_id']
        q_text = ann['question']
        gt_ans = ann.get('multiple_choice_answer', '')
        gt_exp = ann.get('explanation', [''])[0] if ann.get('explanation') else ''
        gt_full = f"{gt_ans} because {gt_exp}" if gt_exp else gt_ans

        img_path = os.path.join(VAL_IMAGE_DIR, f"COCO_val2014_{img_id:012d}.jpg")
        if not os.path.exists(img_path):
            continue

        img_pil    = Image.open(img_path).convert("RGB")
        img_tensor = transform(img_pil)
        q_tensor   = torch.tensor(vocab_q.numericalize(q_text), dtype=torch.long)

        # Get predictions from all loaded models
        preds = {}
        with torch.no_grad():
            for m, model in loaded_models.items():
                if m in ('C', 'D'):
                    pred = greedy_decode_with_attention(model, img_tensor, q_tensor, vocab_a, device=DEVICE)
                else:
                    pred = greedy_decode(model, img_tensor, q_tensor, vocab_a, device=DEVICE)
                preds[m] = pred

        # Display
        fig, ax = plt.subplots(1, 1, figsize=(5, 4))
        ax.imshow(img_pil)
        ax.axis('off')
        ax.set_title(f"Sample {idx+1} — Q: {q_text}", fontsize=10, wrap=True, pad=10)
        plt.tight_layout()
        plt.show()

        print(f"  {'Ground Truth':15s}: {gt_full}")
        for m, pred in sorted(preds.items()):
            marker = '★' if m in ('C', 'D') else ' '
            print(f"  {'Model ' + m:15s}: {pred} {marker}")
        print("─" * 80)

In [ ]:
# ── Greedy vs Beam Search: Side-by-side predictions ───────────────────────────
if not loaded_models:
    print("⚠ No models loaded")
else:
    # Pick best available model (D > C > B > A)
    best_m = next((m for m in 'DCBA' if m in loaded_models), None)
    best_model = loaded_models[best_m]
    has_attn = best_m in ('C', 'D')
    beam_fn = beam_search_decode_with_attention if has_attn else beam_search_decode
    greedy_fn = greedy_decode_with_attention if has_attn else greedy_decode

    print(f"Greedy vs Beam Search (width=5) — Model {best_m}:\n")

    for i, ann in enumerate(samples[:6]):
        img_id = ann['img_id']
        q_text = ann['question']
        gt_ans = ann.get('multiple_choice_answer', '')
        gt_exp = ann.get('explanation', [''])[0] if ann.get('explanation') else ''
        gt_full = f"{gt_ans} because {gt_exp}" if gt_exp else gt_ans

        img_path = os.path.join(VAL_IMAGE_DIR, f"COCO_val2014_{img_id:012d}.jpg")
        if not os.path.exists(img_path):
            continue

        img_pil = Image.open(img_path).convert("RGB")
        img_tensor = transform(img_pil)
        q_tensor = torch.tensor(vocab_q.numericalize(q_text), dtype=torch.long)

        with torch.no_grad():
            greedy_pred = greedy_fn(best_model, img_tensor, q_tensor, vocab_a, device=DEVICE)
            beam_pred = beam_fn(best_model, img_tensor, q_tensor, vocab_a,
                                device=DEVICE, beam_width=5, no_repeat_ngram_size=3)

        print(f"  [{i+1}] Q: {q_text}")
        print(f"       GT    : {gt_full}")
        print(f"       Greedy: {greedy_pred}")
        print(f"       Beam-5: {beam_pred}")
        print()

---
## 7. Attention Visualization (Model C/D)

Model C/D có **49 spatial regions** (7×7 grid). Tại mỗi decode step, attention weights cho biết model đang "nhìn" vào vùng nào.

In [ ]:
# ── Attention Visualization Functions ──────────────────────────────────────────
import torch.nn.functional as F

def decode_with_attention_steps(model, image_tensor, question_tensor,
                                vocab_a, max_len=20, device='cpu'):
    """
    Run greedy decode step by step, collecting attention weights.
    Returns: (tokens: list[str], alphas: list[np.array(49,)])
    """
    with torch.no_grad():
        img      = image_tensor.unsqueeze(0).to(device)
        question = question_tensor.unsqueeze(0).to(device)

        img_features  = model.i_encoder(img)
        img_features  = F.normalize(img_features, p=2, dim=-1)
        question_feat, q_hidden = model.q_encoder(question)

        img_mean = img_features.mean(dim=1)
        fusion   = model.fusion(img_mean, question_feat)

        h_0 = fusion.unsqueeze(0).repeat(model.num_layers, 1, 1)
        c_0 = torch.zeros_like(h_0)
        hidden = (h_0, c_0)

        start_idx = vocab_a.word2idx['<start>']
        end_idx   = vocab_a.word2idx['<end>']
        token     = torch.tensor([[start_idx]], dtype=torch.long).to(device)

        tokens, alphas = [], []
        coverage = None

        for _ in range(max_len):
            logit, hidden, alpha, coverage = model.decoder.decode_step(
                token, hidden, img_features, q_hidden, coverage
            )
            pred = logit.argmax(dim=-1).item()
            if pred == end_idx:
                break
            tokens.append(vocab_a.idx2word.get(pred, '<unk>'))
            alphas.append(alpha.squeeze(0).cpu().numpy())
            token = torch.tensor([[pred]], dtype=torch.long).to(device)

    return tokens, alphas


def visualize_attention_inline(model, img_path, question, vocab_q, vocab_a,
                               device, max_words=10, max_len=25):
    """
    Visualize attention heatmaps overlaid on the image for each generated token.
    """
    img_pil = Image.open(img_path).convert("RGB")
    img_tensor = transform(img_pil)
    q_tensor = torch.tensor(vocab_q.numericalize(question), dtype=torch.long)

    words, attn_weights = decode_with_attention_steps(
        model, img_tensor, q_tensor, vocab_a, device=device, max_len=max_len
    )

    if not words:
        print("No words generated")
        return

    n_show = min(len(words), max_words)
    fig, axes = plt.subplots(1, n_show + 1, figsize=(3 * (n_show + 1), 3.5))

    # Original image
    img_np = np.array(img_pil.resize((224, 224)))
    axes[0].imshow(img_pil)
    axes[0].set_title("Original", fontsize=9)
    axes[0].axis('off')

    for i in range(n_show):
        ax = axes[i + 1]
        attn = attn_weights[i].reshape(7, 7)
        attn_resized = np.array(
            Image.fromarray((attn * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)
        ) / 255.0
        attn_resized = (attn_resized - attn_resized.min()) / (attn_resized.max() - attn_resized.min() + 1e-8)

        ax.imshow(img_np)
        ax.imshow(attn_resized, cmap='jet', alpha=0.5)
        ax.set_title(f'"{words[i]}"', fontsize=9, fontweight='bold')
        ax.axis('off')

    answer = ' '.join(words)
    fig.suptitle(f'Q: {question}\nA: {answer}', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("✅ Attention visualization functions ready")

In [ ]:
# ── Visualize Attention for Multiple Samples ──────────────────────────────────
attn_model_type = next((m for m in 'DC' if m in loaded_models), None)

if attn_model_type is None:
    print("⚠ Model C/D not loaded — attention visualization requires spatial attention models")
else:
    attn_model = loaded_models[attn_model_type]
    print(f"Using Model {attn_model_type} for attention visualization\n")

    for i, ann in enumerate(samples[:6]):
        img_id = ann['img_id']
        q_text = ann['question']
        img_path = os.path.join(VAL_IMAGE_DIR, f"COCO_val2014_{img_id:012d}.jpg")

        if not os.path.exists(img_path):
            print(f"  ⚠ Image not found: {img_path}")
            continue

        print(f"\n--- Sample {i+1} ---")
        visualize_attention_inline(
            attn_model, img_path, q_text, vocab_q, vocab_a, DEVICE
        )
        # Also show ground truth
        gt_ans = ann.get('multiple_choice_answer', '')
        gt_exp = ann.get('explanation', [''])[0] if ann.get('explanation') else ''
        print(f"  GT: {gt_ans} because {gt_exp}")
        print()

---
## 8. Error Analysis

Phân tích chi tiết các trường hợp model dự đoán sai/tốt, theo nhiều chiều:
- Answer length distribution
- Question type analysis (What/Where/How/Is/...)
- Top errors & best predictions
- BLEU score distribution

In [ ]:
# ── Answer Length Analysis ─────────────────────────────────────────────────────
if all_per_sample_greedy:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Prediction length distribution per model
    ax = axes[0]
    for m in sorted(all_per_sample_greedy.keys()):
        pred_lens = [s['pred_len'] for s in all_per_sample_greedy[m]]
        ax.hist(pred_lens, bins=range(0, 50), alpha=0.5, label=f'Model {m}',
                color=MODEL_COLORS.get(m))
    # Ground truth distribution
    gt_lens = [s['gt_len'] for s in list(all_per_sample_greedy.values())[0]]
    ax.hist(gt_lens, bins=range(0, 50), alpha=0.3, label='Ground Truth',
            color='gray', linestyle='--')
    ax.set_title('Answer Length Distribution', fontsize=12, fontweight='bold')
    ax.set_xlabel('Length (tokens)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

    # BLEU-4 vs answer length (scatter)
    ax = axes[1]
    for m in sorted(all_per_sample_greedy.keys()):
        gt_lens = [s['gt_len'] for s in all_per_sample_greedy[m]]
        bleu4s  = [s['bleu4'] for s in all_per_sample_greedy[m]]
        # Bin by length and average
        bins = defaultdict(list)
        for l, b in zip(gt_lens, bleu4s):
            bins[min(l, 30)].append(b)
        sorted_bins = sorted(bins.items())
        lengths = [b[0] for b in sorted_bins]
        avg_bleu = [np.mean(b[1]) for b in sorted_bins]
        ax.plot(lengths, avg_bleu, marker='o', ms=4, label=f'Model {m}',
                color=MODEL_COLORS.get(m), lw=1.5)
    ax.set_title('BLEU-4 vs Ground Truth Length', fontsize=12, fontweight='bold')
    ax.set_xlabel('GT Answer Length (tokens)')
    ax.set_ylabel('Average BLEU-4')
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig('checkpoints/eval_length_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⚠ No per-sample data available")

In [ ]:
# ── Question Type Analysis ────────────────────────────────────────────────────
def get_question_type(question):
    """Extract question type from first word."""
    q = question.strip().lower()
    first_word = q.split()[0] if q.split() else 'other'
    type_map = {
        'what': 'What', 'which': 'What',
        'where': 'Where',
        'how': 'How',
        'is': 'Yes/No', 'are': 'Yes/No', 'was': 'Yes/No', 'were': 'Yes/No',
        'do': 'Yes/No', 'does': 'Yes/No', 'did': 'Yes/No',
        'can': 'Yes/No', 'could': 'Yes/No', 'would': 'Yes/No',
        'who': 'Who',
        'why': 'Why',
    }
    return type_map.get(first_word, 'Other')


if all_per_sample_greedy and val_anns:
    # Get question types for eval samples
    eval_anns = val_anns[:NUM_EVAL_SAMPLES] if NUM_EVAL_SAMPLES else val_anns
    q_types = [get_question_type(ann['question']) for ann in eval_anns]

    # Compute BLEU-4 and METEOR by question type per model
    type_metrics = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))
    for m in sorted(all_per_sample_greedy.keys()):
        for i, (sample, qtype) in enumerate(zip(all_per_sample_greedy[m], q_types)):
            type_metrics[m][qtype]['bleu4'].append(sample['bleu4'])
            type_metrics[m][qtype]['meteor'].append(sample['meteor'])

    # Plot
    q_type_order = ['Yes/No', 'What', 'How', 'Where', 'Who', 'Why', 'Other']
    q_type_order = [qt for qt in q_type_order if any(
        qt in type_metrics[m] for m in type_metrics
    )]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    x = np.arange(len(q_type_order))
    n_models = len(all_per_sample_greedy)
    width = 0.8 / n_models

    for idx, m in enumerate(sorted(all_per_sample_greedy.keys())):
        bleu4_means = [np.mean(type_metrics[m][qt]['bleu4']) if qt in type_metrics[m] else 0
                       for qt in q_type_order]
        meteor_means = [np.mean(type_metrics[m][qt]['meteor']) if qt in type_metrics[m] else 0
                        for qt in q_type_order]

        offset = (idx - n_models/2 + 0.5) * width
        ax1.bar(x + offset, bleu4_means, width, label=f'Model {m}',
                color=MODEL_COLORS.get(m), alpha=0.85)
        ax2.bar(x + offset, meteor_means, width, label=f'Model {m}',
                color=MODEL_COLORS.get(m), alpha=0.85)

    for ax, title in [(ax1, 'BLEU-4 by Question Type'), (ax2, 'METEOR by Question Type')]:
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(q_type_order, fontsize=9)
        ax.legend(fontsize=8)
        ax.set_ylabel('Score')

    # Count per question type
    type_counts = Counter(q_types)
    count_str = ' | '.join([f"{qt}: {type_counts[qt]}" for qt in q_type_order])
    fig.suptitle(f'Performance by Question Type\n({count_str})',
                 fontsize=13, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.savefig('checkpoints/eval_question_type.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⚠ No data for question type analysis")

In [ ]:
# ── BLEU-4 Score Distribution (Histogram) ─────────────────────────────────────
if all_per_sample_greedy:
    fig, axes = plt.subplots(1, len(all_per_sample_greedy), figsize=(5 * len(all_per_sample_greedy), 4))
    if len(all_per_sample_greedy) == 1:
        axes = [axes]

    for ax, m in zip(axes, sorted(all_per_sample_greedy.keys())):
        bleu4s = [s['bleu4'] for s in all_per_sample_greedy[m]]
        ax.hist(bleu4s, bins=50, color=MODEL_COLORS.get(m), alpha=0.8, edgecolor='white')
        ax.axvline(np.mean(bleu4s), color='red', ls='--', lw=1.5, label=f'Mean: {np.mean(bleu4s):.4f}')
        ax.axvline(np.median(bleu4s), color='blue', ls='--', lw=1.5, label=f'Median: {np.median(bleu4s):.4f}')
        ax.set_title(f'Model {m} — BLEU-4 Distribution', fontsize=11, fontweight='bold')
        ax.set_xlabel('BLEU-4')
        ax.set_ylabel('Count')
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig('checkpoints/eval_bleu4_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Top Best & Worst Predictions ──────────────────────────────────────────────
if all_per_sample_greedy:
    best_m = next((m for m in 'DCBA' if m in all_per_sample_greedy), None)
    if best_m:
        samples_sorted = sorted(all_per_sample_greedy[best_m], key=lambda x: x['bleu4'], reverse=True)

        print(f"╔══════════════════════════════════════════════════════════════╗")
        print(f"║         TOP 10 BEST Predictions (Model {best_m})               ║")
        print(f"╚══════════════════════════════════════════════════════════════╝")
        for i, s in enumerate(samples_sorted[:10]):
            print(f"  {i+1:2d}. BLEU-4={s['bleu4']:.4f} | METEOR={s['meteor']:.4f}")
            print(f"      GT  : {s['ground_truth'][:80]}")
            print(f"      Pred: {s['prediction'][:80]}")
            print()

        print(f"\n╔══════════════════════════════════════════════════════════════╗")
        print(f"║         TOP 10 WORST Predictions (Model {best_m})              ║")
        print(f"╚══════════════════════════════════════════════════════════════╝")
        for i, s in enumerate(samples_sorted[-10:]):
            print(f"  {i+1:2d}. BLEU-4={s['bleu4']:.4f} | METEOR={s['meteor']:.4f}")
            print(f"      GT  : {s['ground_truth'][:80]}")
            print(f"      Pred: {s['prediction'][:80]}")
            print()

---
## 9. Statistical Analysis

- Paired t-test để kiểm tra sự khác biệt có ý nghĩa thống kê giữa các models
- 95% Confidence intervals cho metrics chính

In [ ]:
# ── Paired t-test between models ──────────────────────────────────────────────
from scipy import stats

if len(all_per_sample_greedy) >= 2:
    models_list = sorted(all_per_sample_greedy.keys())
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║          Paired t-test (BLEU-4) Between Models             ║")
    print("╚══════════════════════════════════════════════════════════════╝")
    print(f"\n  {'Comparison':20s}  {'t-stat':>10s}  {'p-value':>10s}  {'Significant?':>14s}")
    print("  " + "─" * 60)

    for i in range(len(models_list)):
        for j in range(i+1, len(models_list)):
            m1, m2 = models_list[i], models_list[j]
            s1 = [s['bleu4'] for s in all_per_sample_greedy[m1]]
            s2 = [s['bleu4'] for s in all_per_sample_greedy[m2]]
            t_stat, p_val = stats.ttest_rel(s1, s2)
            sig = '✅ Yes (p<0.05)' if p_val < 0.05 else '❌ No'
            print(f"  {m1} vs {m2}:{'':12s}  {t_stat:>10.4f}  {p_val:>10.6f}  {sig:>14s}")

    print()

    # Same for METEOR
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║          Paired t-test (METEOR) Between Models             ║")
    print("╚══════════════════════════════════════════════════════════════╝")
    print(f"\n  {'Comparison':20s}  {'t-stat':>10s}  {'p-value':>10s}  {'Significant?':>14s}")
    print("  " + "─" * 60)

    for i in range(len(models_list)):
        for j in range(i+1, len(models_list)):
            m1, m2 = models_list[i], models_list[j]
            s1 = [s['meteor'] for s in all_per_sample_greedy[m1]]
            s2 = [s['meteor'] for s in all_per_sample_greedy[m2]]
            t_stat, p_val = stats.ttest_rel(s1, s2)
            sig = '✅ Yes (p<0.05)' if p_val < 0.05 else '❌ No'
            print(f"  {m1} vs {m2}:{'':12s}  {t_stat:>10.4f}  {p_val:>10.6f}  {sig:>14s}")
else:
    print("⚠ Need at least 2 models for paired t-test")

In [ ]:
# ── 95% Confidence Intervals ──────────────────────────────────────────────────
if all_per_sample_greedy:
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║       95% Confidence Intervals (Bootstrap, n=1000)         ║")
    print("╚══════════════════════════════════════════════════════════════╝\n")

    def bootstrap_ci(scores, n_bootstrap=1000, ci=0.95):
        """Compute bootstrap confidence interval."""
        means = []
        for _ in range(n_bootstrap):
            sample = np.random.choice(scores, size=len(scores), replace=True)
            means.append(np.mean(sample))
        lower = np.percentile(means, (1-ci)/2 * 100)
        upper = np.percentile(means, (1+ci)/2 * 100)
        return np.mean(scores), lower, upper

    for m in sorted(all_per_sample_greedy.keys()):
        print(f"  Model {m}:")
        for metric in ['bleu4', 'meteor']:
            scores = [s[metric] for s in all_per_sample_greedy[m]]
            mean, lower, upper = bootstrap_ci(scores)
            metric_name = 'BLEU-4' if metric == 'bleu4' else 'METEOR'
            print(f"    {metric_name:10s}: {mean:.4f}  [{lower:.4f}, {upper:.4f}]")
        if HAS_BERTSCORE and 'bertscore' in all_per_sample_greedy[m][0]:
            scores = [s['bertscore'] for s in all_per_sample_greedy[m]]
            mean, lower, upper = bootstrap_ci(scores)
            print(f"    {'BERTScore':10s}: {mean:.4f}  [{lower:.4f}, {upper:.4f}]")
        print()

    # Visualization: CI plot
    fig, ax = plt.subplots(figsize=(10, 5))
    metrics_for_ci = ['bleu4', 'meteor']
    offsets = np.linspace(-0.15, 0.15, len(all_per_sample_greedy))

    for idx, m in enumerate(sorted(all_per_sample_greedy.keys())):
        means_list, lowers, uppers = [], [], []
        for metric in metrics_for_ci:
            scores = [s[metric] for s in all_per_sample_greedy[m]]
            mean, lower, upper = bootstrap_ci(scores)
            means_list.append(mean)
            lowers.append(mean - lower)
            uppers.append(upper - mean)

        x_pos = np.arange(len(metrics_for_ci)) + offsets[idx]
        ax.errorbar(x_pos, means_list, yerr=[lowers, uppers], fmt='o', capsize=5,
                    color=MODEL_COLORS.get(m), label=f'Model {m}', markersize=8, lw=2)

    ax.set_xticks(range(len(metrics_for_ci)))
    ax.set_xticklabels(['BLEU-4', 'METEOR'], fontsize=11)
    ax.set_title('95% Confidence Intervals (Bootstrap)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Score')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig('checkpoints/eval_confidence_intervals.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 10. Summary & Conclusions

In [ ]:
# ── Final Summary Report ──────────────────────────────────────────────────────
print("=" * 70)
print("  FINAL EVALUATION REPORT — VQA-E Generative Models")
print("=" * 70)
print()

# Best model determination
if all_metrics_greedy:
    best_bleu4_model = max(all_metrics_greedy.items(), key=lambda x: x[1]['bleu4'])
    best_meteor_model = max(all_metrics_greedy.items(), key=lambda x: x[1]['meteor'])

    print("  📊 GREEDY DECODE RESULTS:")
    print("  " + "-" * 66)
    print(f"  {'Model':<10} {'BLEU-4':>8} {'METEOR':>8} {'BERTScore':>10} {'ExactMatch':>12}")
    print("  " + "-" * 66)
    for m in 'ABCD':
        if m in all_metrics_greedy:
            r = all_metrics_greedy[m]
            mark = ' ← BEST' if m == best_bleu4_model[0] else ''
            bs = f"{r['bertscore']:.4f}" if HAS_BERTSCORE else 'N/A'
            print(f"  Model {m:<5} {r['bleu4']:>8.4f} {r['meteor']:>8.4f} {bs:>10} {r['exact_match']:>10.2f}%{mark}")
    print()

if all_metrics_beam:
    best_beam_model = max(all_metrics_beam.items(), key=lambda x: x[1]['bleu4'])
    print(f"  📊 BEAM SEARCH (width={BEAM_WIDTH}) RESULTS:")
    print("  " + "-" * 66)
    print(f"  {'Model':<10} {'BLEU-4':>8} {'METEOR':>8} {'BERTScore':>10} {'ExactMatch':>12}")
    print("  " + "-" * 66)
    for m in 'ABCD':
        if m in all_metrics_beam:
            r = all_metrics_beam[m]
            mark = ' ← BEST' if m == best_beam_model[0] else ''
            bs = f"{r['bertscore']:.4f}" if HAS_BERTSCORE else 'N/A'
            print(f"  Model {m:<5} {r['bleu4']:>8.4f} {r['meteor']:>8.4f} {bs:>10} {r['exact_match']:>10.2f}%{mark}")
    print()

# Key findings
print("  🔍 KEY FINDINGS:")
print("  " + "─" * 66)
if all_metrics_greedy:
    print(f"  • Best overall model (BLEU-4): Model {best_bleu4_model[0]} ({best_bleu4_model[1]['bleu4']:.4f})")
    print(f"  • Best overall model (METEOR): Model {best_meteor_model[0]} ({best_meteor_model[1]['meteor']:.4f})")

if all_metrics_greedy and all_metrics_beam:
    # Beam search improvement
    for m in sorted(set(all_metrics_greedy.keys()) & set(all_metrics_beam.keys())):
        delta = all_metrics_beam[m]['bleu4'] - all_metrics_greedy[m]['bleu4']
        print(f"  • Beam Search improvement for Model {m}: {delta:+.4f} BLEU-4")

if histories:
    print()
    for m, h in sorted(histories.items()):
        n_ep = len(h['train_loss'])
        best_val = min(h['val_loss'])
        best_ep = h['val_loss'].index(best_val) + 1
        print(f"  • Model {m}: {n_ep} epochs trained, best val loss {best_val:.4f} at epoch {best_ep}")

print()
print("=" * 70)
print("  ✅ Evaluation complete!")
print("=" * 70)

In [ ]:
# ── Export Results to JSON ─────────────────────────────────────────────────────
results_export = {
    'greedy': {m: r for m, r in all_metrics_greedy.items()},
    'beam_search': {m: r for m, r in all_metrics_beam.items()} if all_metrics_beam else {},
    'config': {
        'num_eval_samples': NUM_EVAL_SAMPLES,
        'beam_width': BEAM_WIDTH,
        'no_repeat_ngram': NO_REPEAT_NGRAM,
        'device': str(DEVICE),
    },
    'model_info': model_info,
}

export_path = 'checkpoints/evaluation_results.json'
with open(export_path, 'w') as f:
    json.dump(results_export, f, indent=2)
print(f"✅ Results exported to {export_path}")

In [ ]:
# ── List all generated evaluation artifacts ───────────────────────────────────
eval_files = [
    'checkpoints/eval_comparison_greedy.png',
    'checkpoints/eval_greedy_vs_beam.png',
    'checkpoints/eval_training_curves.png',
    'checkpoints/eval_length_analysis.png',
    'checkpoints/eval_question_type.png',
    'checkpoints/eval_bleu4_distribution.png',
    'checkpoints/eval_confidence_intervals.png',
    'checkpoints/evaluation_results.json',
]

print("📁 Generated evaluation artifacts:")
for f in eval_files:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f"  ✅ {f} ({size:.0f} KB)")
    else:
        print(f"  ⬜ {f} (not yet generated)")